# Tutorial 10: Long-Running Workflows & Channels

**Difficulty**: Advanced | **Time**: 60 minutes

## Learning Objectives

After completing this tutorial, you will be able to:

- Understand long-running vs. sequential execution modes
- Work with channels for inter-node communication
- Implement continuous workers with mailboxes
- Handle shutdown signals gracefully
- Build reactive event-driven workflows
- Manage backpressure and message queuing

## Prerequisites

Completion of Tutorials 1-7 (Foundation) is required. You should understand:
- Basic node and graph concepts (Tutorial 1)
- Graph connections and flow control (Tutorial 3)
- Agent fundamentals (Tutorial 5)

## Real-World Scenarios

- Real-time data processing pipelines
- Message queue consumers with multiple workers
- Streaming analytics workflows
- Continuous monitoring systems
- Event-driven microservices

## Tutorial Structure

This tutorial is divided into 5 sections:

1. **Understanding Long-Running Workflows** - Core concepts and differences
2. **Channel System** - Message passing and communication patterns
3. **Mailbox Persistence** - Durable message storage and recovery
4. **Advanced Patterns** - Priority routing, batching, and streaming
5. **Practical Examples** - Real-world use cases and implementations

Let's get started!

## Section 1: Understanding Long-Running Workflows

### Sequential vs. Long-Running Execution

In standard Spark workflows (Tutorials 1-7), execution is **sequential**:
- Nodes execute one after another
- Messages are passed directly between nodes
- Graph `run()` waits for complete workflow to finish
- Single execution path from start to end

In **long-running workflows**, execution is **concurrent**:
- All nodes run continuously as workers
- Nodes communicate via **channels** (message queues)
- Nodes process messages from their **mailbox** (a channel)
- Graph `run()` manages all concurrent node workers
- Execution continues until timeout or manual shutdown

### Key Differences

| Aspect | Sequential Workflow | Long-Running Workflow |
|---------|-------------------|----------------------|
| Execution | Node A → Node B → Node C | All nodes run concurrently |
| Message passing | Direct method calls | Channel-based messaging |
| State | Ephemeral, per-execution | Persistent via GraphState |
| Duration | One-time completion | Continuous processing |
| Use case | Data transformation pipelines | Real-time processing systems |

### TaskType Enumeration

The `TaskType` enum defines different execution modes:

```python
from spark.graphs import TaskType

# Standard sequential execution (default)
TaskType.ONE_OFF  

# Continuous concurrent execution
TaskType.LONG_RUNNING

# Streaming data processing
TaskType.STREAMING

# Resumable long-running tasks
TaskType.RESUMABLE
```

In [ ]:
# Example: Basic Long-Running Workflow
from spark.graphs import Graph, Task, TaskType
from spark.nodes import Node
from spark.nodes.types import NodeMessage
import asyncio

class ProducerNode(Node):
    """Continuously generates data."""
    def __init__(self, interval=1.0, max_messages=5):
        super().__init__()
        self.interval = interval
        self.max_messages = max_messages
        self.count = 0

    async def process(self, context):
        print(f"Producer starting...")
        
        while not self._stop_flag and self.count < self.max_messages:
            # Create data
            data = {'id': self.count + 1, 'value': f'item_{self.count + 1}'}
            print(f"Producer created: {data}")
            
            # Return data to next node (goes to channel)
            yield NodeMessage(content=data)
            
            await asyncio.sleep(self.interval)
            self.count += 1
        
        print(f"Producer finished after {self.count} messages")

class ConsumerNode(Node):
    """Consumes data continuously."""
    def __init__(self):
        super().__init__()
        self.processed = 0

    async def process(self, context):
        data = context.inputs.content
        self.processed += 1
        print(f"Consumer processed: {data} (total: {self.processed})")

# Create nodes
producer = ProducerNode(interval=0.5, max_messages=3)
consumer = ConsumerNode()

# Connect nodes
producer >> consumer

# Create graph
graph = Graph(start=producer)

# Create LONG_RUNNING task
task = Task(
    inputs=NodeMessage(content={'start': True}),
    type=TaskType.LONG_RUNNING,
    budget={'max_seconds': 5}  # Run for 5 seconds
)

# This will run both nodes concurrently for up to 5 seconds
# result = await graph.run(task)

### Key Observations

1. **Concurrent Execution**: Both `ProducerNode` and `ConsumerNode` run simultaneously
2. **Message Passing**: Producer's output goes to Consumer's mailbox (channel)
3. **Continuous Loop**: Nodes keep running until stopped or timeout
4. **Yield Pattern**: Producer uses `yield` to send multiple messages
5. **Budget Control**: Task budget controls overall execution time

### Exercise 1: Basic Long-Running Workflow

Create a long-running workflow with:
1. A `CounterNode` that produces incrementing numbers every 0.5 seconds
2. An `EvenFilterNode` that only passes even numbers
3. A `SumNode` that keeps a running sum of received numbers

Configure it to run for 3 seconds and see the results!

In [ ]:
# Your solution for Exercise 1
from spark.graphs import Graph, Task, TaskType
from spark.nodes import Node
from spark.nodes.types import NodeMessage
import asyncio

class CounterNode(Node):
    def __init__(self, interval=0.5, max_count=10):
        super().__init__()
        self.interval = interval
        self.max_count = max_count
        self.count = 0

    async def process(self, context):
        # TODO: Implement counting logic
        pass

class EvenFilterNode(Node):
    async def process(self, context):
        # TODO: Implement even number filtering
        pass

class SumNode(Node):
    def __init__(self):
        super().__init__()
        self.total = 0

    async def process(self, context):
        # TODO: Implement sum accumulation
        pass

# TODO: Create nodes and connect them
# TODO: Create LONG_RUNNING task with 3-second budget
# TODO: Run the workflow

<details>
<summary>Click to see solution for Exercise 1</summary>

```python
class CounterNode(Node):
    def __init__(self, interval=0.5, max_count=10):
        super().__init__()
        self.interval = interval
        self.max_count = max_count
        self.count = 0

    async def process(self, context):
        while not self._stop_flag and self.count < self.max_count:
            self.count += 1
            print(f"Counter: {self.count}")
            yield NodeMessage(content={'number': self.count})
            await asyncio.sleep(self.interval)

class EvenFilterNode(Node):
    async def process(self, context):
        number = context.inputs.content.get('number')
        if number % 2 == 0:
            print(f"EvenFilter: {number} is even, passing through")
            return context.inputs
        else:
            print(f"EvenFilter: {number} is odd, blocking")
            return None

class SumNode(Node):
    def __init__(self):
        super().__init__()
        self.total = 0

    async def process(self, context):
        number = context.inputs.content.get('number')
        self.total += number
        print(f"SumNode: Added {number}, total = {self.total}")

# Create and connect nodes
counter = CounterNode(interval=0.3, max_count=10)
filter_node = EvenFilterNode()
sum_node = SumNode()

counter >> filter_node
filter_node >> sum_node

# Create graph and task
graph = Graph(start=counter)
task = Task(
    inputs=NodeMessage(content={'start': True}),
    type=TaskType.LONG_RUNNING,
    budget={'max_seconds': 3}
)

# Run workflow
# result = await graph.run(task)
```

</details>

## Section 2: Channel System

### Introduction to Channels

In long-running workflows, nodes communicate through **channels** instead of direct method calls. Channels provide:

1. **Asynchronous Communication**: Non-blocking message passing
2. **Buffering**: Queue messages when consumers are busy
3. **Backpressure**: Prevent overwhelming downstream nodes
4. **Persistence**: Durable message storage across restarts
5. **Metadata**: Rich context information with messages

### ChannelMessage Structure

Every message sent through a channel is wrapped in a `ChannelMessage`:

```python
@dataclass
class ChannelMessage:
    payload: Any              # Actual message content
    metadata: dict[str, Any]   # Context information
    ack: Optional[Callable]    # Acknowledgment callback
    is_shutdown: bool = False   # Shutdown signal flag
```

### Channel Types

#### InMemoryChannel

Basic in-memory channel using asyncio.Queue:

```python
from spark.nodes.channels import InMemoryChannel, ChannelMessage

# Create channel with size limit
channel = InMemoryChannel(maxsize=100, name="data_stream")

# Send message
await channel.send(ChannelMessage(
    payload={'data': 'hello'},
    metadata={'source': 'producer', 'priority': 'high'}
))

# Receive message
message = await channel.receive()
print(message.payload)  # {'data': 'hello'}
print(message.metadata)  # {'source': 'producer', 'priority': 'high'}
```

In [ ]:
# Example: Direct Channel Usage
from spark.nodes.channels import InMemoryChannel, ChannelMessage
import asyncio

async def demo_basic_channel():
    # Create channel
    channel = InMemoryChannel(name="demo", maxsize=5)
    
    # Producer task
    async def producer():
        for i in range(5):
            message = ChannelMessage(
                payload={'id': i, 'value': f'message_{i}'},
                metadata={'producer': 'demo_producer', 'timestamp': asyncio.get_event_loop().time()}
            )
            print(f"Producer sending: {message.payload}")
            await channel.send(message)
            await asyncio.sleep(0.1)
        
        # Send shutdown signal
        shutdown_msg = ChannelMessage(
            payload=None,
            is_shutdown=True,
            metadata={'reason': 'producer_finished'}
        )
        await channel.send(shutdown_msg)
    
    # Consumer task
    async def consumer():
        while True:
            message = await channel.receive()
            
            if message.is_shutdown:
                print(f"Consumer received shutdown: {message.metadata}")
                break
            
            print(f"Consumer received: {message.payload} (metadata: {message.metadata['producer']})")
    
    # Run producer and consumer concurrently
    await asyncio.gather(producer(), consumer())

# Run the demo
# await demo_basic_channel()

### Node Mailboxes

In long-running workflows, each node automatically gets a **mailbox** - a channel where it receives messages:

```python
class MyNode(Node):
    async def process(self, context):
        # Message comes from node.mailbox via context.inputs
        message = context.inputs.content
        
        # Process message...
        
        # Return None to not forward, or return data for next nodes
        return processed_data
```

### ForwardingChannel

For message enrichment and monitoring:

```python
from spark.nodes.channels import ForwardingChannel

# Create base channel
base_channel = InMemoryChannel(name="messages")

# Wrap with forwarding for enrichment
forwarding = ForwardingChannel(
    downstream=base_channel,
    name="enriched_messages",
    metadata_defaults={
        'workflow_id': 'demo_001',
        'processed_at': lambda: time.time(),
        'environment': 'production'
    }
)

# Messages sent to forwarding get enriched
await forwarding.send(ChannelMessage(payload={'data': 'test'}))
# Received message will have enriched metadata
```

### Exercise 2: Channel Communication Patterns

Implement different communication patterns:

1. **Fan-Out**: One producer sends to multiple consumers
2. **Fan-In**: Multiple producers send to one aggregator
3. **Priority Queue**: High priority messages are processed first

Use channels directly (not graphs) to demonstrate these patterns.

In [ ]:
# Your solution for Exercise 2
from spark.nodes.channels import InMemoryChannel, ChannelMessage, ForwardingChannel
import asyncio

async def demo_fan_out():
    """TODO: One producer sends to multiple consumers"""
    pass

async def demo_fan_in():
    """TODO: Multiple producers send to one aggregator"""
    pass

async def demo_priority_queue():
    """TODO: Priority-based message processing"""
    pass

# TODO: Test your implementations

<details>
<summary>Click to see solution for Exercise 2</summary>

```python
async def demo_fan_out():
    """One producer sends to multiple consumers"""
    print("=== Fan-Out Pattern ===")
    
    # Create channels for each consumer
    consumer1_channel = InMemoryChannel(name="consumer1")
    consumer2_channel = InMemoryChannel(name="consumer2")
    
    async def producer():
        for i in range(5):
            message = ChannelMessage(payload={'id': i, 'data': f'broadcast_{i}'})
            print(f"Producer broadcasting: {message.payload}")
            
            # Send to all consumers
            await consumer1_channel.send(message)
            await consumer2_channel.send(message)
            await asyncio.sleep(0.1)
    
    async def consumer(name, channel):
        for i in range(5):
            message = await channel.receive()
            print(f"{name} received: {message.payload}")
    
    # Run all tasks
    await asyncio.gather(
        producer(),
        consumer("Consumer1", consumer1_channel),
        consumer("Consumer2", consumer2_channel)
    )

async def demo_fan_in():
    """Multiple producers send to one aggregator"""
    print("=== Fan-In Pattern ===")
    
    aggregator_channel = InMemoryChannel(name="aggregator")
    
    async def producer(name, data_list):
        for data in data_list:
            message = ChannelMessage(
                payload={'producer': name, 'data': data},
                metadata={'producer_id': name}
            )
            print(f"{name} sending: {data}")
            await aggregator_channel.send(message)
            await asyncio.sleep(0.05)
    
    async def aggregator():
        received = []
        for _ in range(8):  # Expect 8 messages total
            message = await aggregator_channel.receive()
            received.append(message.payload)
            print(f"Aggregator received: {message.payload}")
        
        print(f"Aggregator total received: {len(received)} messages")
    
    # Run tasks
    await asyncio.gather(
        producer("Producer1", ["A", "B", "C"]),
        producer("Producer2", ["X", "Y", "Z"]),
        producer("Producer3", ["1", "2"]),
        aggregator()
    )

async def demo_priority_queue():
    """Priority-based message processing"""
    print("=== Priority Queue Pattern ===")
    
    # Create separate channels for different priorities
    high_channel = InMemoryChannel(name="high_priority", maxsize=2)
    normal_channel = InMemoryChannel(name="normal_priority", maxsize=5)
    
    async def producer():
        messages = [
            ('high', 'urgent_task'),
            ('normal', 'regular_task_1'),
            ('high', 'critical_update'),
            ('normal', 'regular_task_2'),
            ('high', 'emergency_stop')
        ]
        
        for priority, data in messages:
            message = ChannelMessage(
                payload=data,
                metadata={'priority': priority, 'timestamp': asyncio.get_event_loop().time()}
            )
            
            target_channel = high_channel if priority == 'high' else normal_channel
            print(f"Producer sending {priority} priority: {data}")
            await target_channel.send(message)
            await asyncio.sleep(0.1)
    
    async def consumer():
        received = []
        
        while len(received) < 5:  # Expect 5 messages total
            # Try high priority first
            if not high_channel.empty():
                message = await high_channel.receive()
                print(f"Consumer received HIGH: {message.payload}")
            elif not normal_channel.empty():
                message = await normal_channel.receive()
                print(f"Consumer received NORMAL: {message.payload}")
            else:
                await asyncio.sleep(0.01)  # Brief wait for messages
                continue
            
            received.append(message.payload)
    
    await asyncio.gather(producer(), consumer())
```

</details>

## Section 3: Mailbox Persistence

### Persistent Mailboxes

In production long-running workflows, mailboxes can be persisted to survive:

- **Process Restarts**: Messages survive node/process crashes
- **System Reboots**: Workflow resumes from persisted state
- **Maintenance**: Can stop and restart without losing data

### PersistentMailbox Implementation

```python
from spark.graphs.mailbox import PersistentMailbox

# Persistent mailbox uses GraphState for storage
mailbox = PersistentMailbox(
    graph_state=graph.state,
    mailbox_id="producer_messages",
    maxsize=1000
)

# Messages are automatically persisted to GraphState backend
await mailbox.send(ChannelMessage(payload={'important': 'data'}))
```

### Graph State Backend for Persistence

```python
from spark.graphs import SQLiteStateBackend, Graph

# Use persistent backend for mailbox storage
backend = SQLiteStateBackend("workflow_mailboxes.db")

graph = Graph(
    start=my_node,
    state_backend=backend  # Enables persistent mailboxes
)

# In LONG_RUNNING mode, mailboxes automatically use persistence
task = Task(
    inputs=NodeMessage(content={'start': True}),
    type=TaskType.LONG_RUNNING
)
```

In [ ]:
# Example: Persistent Long-Running Workflow
from spark.graphs import Graph, Task, TaskType, SQLiteStateBackend
from spark.nodes import Node
from spark.nodes.types import NodeMessage
from pathlib import Path
import time

class PersistentProducer(Node):
    def __init__(self, interval=0.5, max_messages=10):
        super().__init__()
        self.interval = interval
        self.max_messages = max_messages
        self.count = 0

    async def process(self, context):
        print("PersistentProducer started")
        
        while not self._stop_flag and self.count < self.max_messages:
            self.count += 1
            data = {
                'id': self.count,
                'content': f'persistent_message_{self.count}',
                'timestamp': time.time()
            }
            
            print(f"PersistentProducer: {data}")
            
            # Update graph state (also persisted)
            await context.graph_state.set('last_produced_id', self.count)
            await context.graph_state.set('total_produced', self.count)
            
            yield NodeMessage(content=data)
            await asyncio.sleep(self.interval)
        
        print(f"PersistentProducer finished after {self.count} messages")

class PersistentConsumer(Node):
    def __init__(self):
        super().__init__()
        self.processed = 0

    async def process(self, context):
        data = context.inputs.content
        self.processed += 1
        
        print(f"PersistentConsumer: {data} (processed: {self.processed})")
        
        # Update graph state
        await context.graph_state.set('last_processed_id', data['id'])
        await context.graph_state.set('total_processed', self.processed)
        
        # Stop after processing message 7
        if data['id'] >= 7:
            print("PersistentConsumer: Reached target, stopping...")
            # This will cause graceful shutdown

# Create persistent backend
db_path = Path("tutorial_10_persistent.db")
if db_path.exists():
    db_path.unlink()  # Clean start

backend = SQLiteStateBackend(db_path)

# Create nodes
producer = PersistentProducer(interval=0.3, max_messages=10)
consumer = PersistentConsumer()

# Connect nodes
producer >> consumer

# Create graph with persistence
graph = Graph(
    start=producer,
    state_backend=backend,
    initial_state={
        'workflow_name': 'persistent_demo',
        'start_time': time.time(),
        'target_processed': 7
    }
)

# Create task
task = Task(
    inputs=NodeMessage(content={'start': True}),
    type=TaskType.LONG_RUNNING,
    budget={'max_seconds': 8},
    task_id="persistent_demo_001"
)

# Run workflow
# result = await graph.run(task)

# Check persisted state
# final_state = await graph.state.get_all()
# print(f"Final persisted state: {final_state}")
# print(f"Database file: {db_path}")

### Exercise 3: Persistent Workflow with Recovery

Create a persistent workflow that can recover from interruption:

1. A `TaskProducer` that creates tasks with IDs 1-20
2. A `TaskProcessor` that processes tasks with delays
3. Use `SQLiteStateBackend` for persistence
4. Demonstrate recovery by interrupting and resuming

Track progress in GraphState and show recovery capabilities.

In [ ]:
# Your solution for Exercise 3
from spark.graphs import Graph, Task, TaskType, SQLiteStateBackend
from spark.nodes import Node
from spark.nodes.types import NodeMessage
from pathlib import Path
import time

class TaskProducer(Node):
    # TODO: Implement task production with persistence
    pass

class TaskProcessor(Node):
    # TODO: Implement task processing with state tracking
    pass

async def run_persistent_workflow_with_recovery():
    # TODO: Create persistent workflow and demonstrate recovery
    pass

# TODO: Test your implementation

<details>
<summary>Click to see solution for Exercise 3</summary>

```python
class TaskProducer(Node):
    def __init__(self, interval=0.4, max_tasks=20):
        super().__init__()
        self.interval = interval
        self.max_tasks = max_tasks
        self.next_id = 1

    async def process(self, context):
        # Resume from last produced ID
        last_id = await context.graph_state.get('last_produced_id', 0)
        self.next_id = last_id + 1
        
        print(f"TaskProducer: Resuming from ID {self.next_id}")
        
        while not self._stop_flag and self.next_id <= self.max_tasks:
            task = {
                'id': self.next_id,
                'type': 'data_processing',
                'payload': f'task_data_{self.next_id}',
                'created_at': time.time()
                'priority': 'high' if self.next_id % 5 == 0 else 'normal'
            }
            
            print(f"TaskProducer: Created task {self.next_id}")
            
            # Update persistent state
            await context.graph_state.set('last_produced_id', self.next_id)
            await context.graph_state.set('total_created', self.next_id)
            
            yield NodeMessage(content=task)
            await asyncio.sleep(self.interval)
            self.next_id += 1
        
        print(f"TaskProducer: Finished producing tasks (last ID: {self.next_id - 1})")

class TaskProcessor(Node):
    def __init__(self, processing_delay=0.6):
        super().__init__()
        self.processing_delay = processing_delay
        self.processed_ids = set()

    async def process(self, context):
        task = context.inputs.content
        task_id = task['id']
        
        # Skip already processed tasks (recovery scenario)
        if task_id in self.processed_ids:
            print(f"TaskProcessor: Skipping already processed task {task_id}")
            return None
        
        # Load processed tasks from state
        processed_state = await context.graph_state.get('processed_task_ids', [])
        self.processed_ids = set(processed_state)
        
        if task_id in self.processed_ids:
            print(f"TaskProcessor: Task {task_id} already processed (from state)")
            return None
        
        print(f"TaskProcessor: Processing task {task_id} ({task['type']})")
        
        # Simulate processing with potential for interruption
        await asyncio.sleep(self.processing_delay)
        
        # Mark as processed
        self.processed_ids.add(task_id)
        await context.graph_state.set('processed_task_ids', list(self.processed_ids))
        await context.graph_state.set('total_processed', len(self.processed_ids))
        await context.graph_state.set('last_processed_task', task)
        
        print(f"TaskProcessor: Completed task {task_id} (total: {len(self.processed_ids)})")
        
        # Stop after processing 15 tasks (demonstrates early completion)
        if len(self.processed_ids) >= 15:
            print(f"TaskProcessor: Reached target of 15 tasks, stopping")

async def run_persistent_workflow_with_recovery():
    db_path = Path("recovery_demo.db")
    
    # First run (might be interrupted)
    print("=== First Run (simulate interruption) ===")
    backend = SQLiteStateBackend(db_path)
    
    producer = TaskProducer(interval=0.3, max_tasks=20)
    processor = TaskProcessor(processing_delay=0.5)
    
    producer >> processor
    
    graph = Graph(
        start=producer,
        state_backend=backend,
        initial_state={
            'workflow_start': time.time(),
            'run_number': 1
        }
    )
    
    task = Task(
        inputs=NodeMessage(content={'start': True}),
        type=TaskType.LONG_RUNNING,
        budget={'max_seconds': 4},  # Short run to simulate interruption
        task_id="recovery_demo_run1"
    )
    
    try:
        await graph.run(task)
    except asyncio.TimeoutError:
        print("First run timed out (simulating interruption)")
    
    # Show state after interruption
    state1 = await backend.get_all()
    print(f"State after interruption: {state1}")
    
    # Second run (recovery)
    print("\n=== Second Run (recovery) ===")
    backend2 = SQLiteStateBackend(db_path)
    
    producer2 = TaskProducer(interval=0.3, max_tasks=20)
    processor2 = TaskProcessor(processing_delay=0.5)
    
    producer2 >> processor2
    
    graph2 = Graph(
        start=producer2,
        state_backend=backend2,
        initial_state={
            'workflow_start': time.time(),
            'run_number': 2
        }
    )
    
    task2 = Task(
        inputs=NodeMessage(content={'resume': True}),
        type=TaskType.LONG_RUNNING,
        budget={'max_seconds': 6},  # Longer run for completion
        task_id="recovery_demo_run2"
    )
    
    await graph2.run(task2)
    
    # Show final state
    final_state = await backend2.get_all()
    print(f"\nFinal state after recovery: {final_state}")

# Run the demo
# await run_persistent_workflow_with_recovery()
```

</details>

## Section 4: Advanced Patterns

### Streaming Data Processing

For continuous data streams, use the `yield` pattern in nodes:

```python
class StreamingNode(Node):
    async def process(self, context):
        while not self._stop_flag:
            # Generate streaming data
            data = fetch_next_data()
            
            # Yield for immediate forwarding to next nodes
            yield NodeMessage(content=data)
            
            # Brief delay to prevent overwhelming downstream
            await asyncio.sleep(0.1)
```

### Backpressure Management

Handle scenarios where producers are faster than consumers:

```python
class BackpressureAwareProducer(Node):
    async def process(self, context):
        while not self._stop_flag:
            data = produce_data()
            
            # Check if downstream mailbox is getting full
            for edge in self.iter_active_edges():
                target_node = edge.to_node
                if hasattr(target_node, 'mailbox'):
                    mailbox = target_node.mailbox
                    if hasattr(mailbox, 'maxsize') and mailbox.maxsize > 0:
                        current_size = len(mailbox._queue) if hasattr(mailbox, '_queue') else 0
                        if current_size >= mailbox.maxsize * 0.8:  # 80% full
                            print(f"Backpressure detected, pausing production")
                            await asyncio.sleep(0.5)
                            continue
            
            yield NodeMessage(content=data)
```

### Graceful Shutdown Patterns

Implement clean shutdown with proper resource cleanup:

```python
class CleanShutdownNode(Node):
    async def process(self, context):
        # Setup cleanup
        cleanup_registered = False
        
        try:
            while not self._stop_flag:
                # Main processing loop
                data = await get_work()
                
                if not cleanup_registered:
                    # Register cleanup function
                    asyncio.get_event_loop().add_signal_handler(
                        signal.SIGTERM, self._handle_shutdown_signal
                    )
                    cleanup_registered = True
                
                result = process_data(data)
                yield NodeMessage(content=result)
        
        finally:
            # Cleanup resources
            print(f"{self.__class__.__name__}: Cleaning up resources")
            await self._cleanup_resources()
    
    async def _cleanup_resources(self):
        # Close file handles, database connections, etc.
        pass
    
    def _handle_shutdown_signal(self, signum, frame):
        print(f"{self.__class__.__name__}: Received shutdown signal")
        self.stop()  # Signal node to stop
```

### Exercise 4: Advanced Long-Running Workflow

Create a sophisticated long-running workflow with:

1. **Rate-Limited Producer**: Produces at configurable rate
2. **Batch Processor**: Processes messages in batches of configurable size
3. **Health Monitor**: Monitors system health and sends alerts
4. **Graceful Shutdown**: Handles SIGTERM/SIGINT signals

Include backpressure management and demonstrate all patterns covered.

In [ ]:
# Your solution for Exercise 4
from spark.graphs import Graph, Task, TaskType
from spark.nodes import Node
from spark.nodes.types import NodeMessage
from spark.nodes.channels import ChannelMessage
import asyncio
import signal
import time

class RateLimitedProducer(Node):
    # TODO: Implement rate-limited production
    pass

class BatchProcessor(Node):
    # TODO: Implement batch processing
    pass

class HealthMonitor(Node):
    # TODO: Implement health monitoring
    pass

async def demo_advanced_workflow():
    # TODO: Create and run advanced workflow
    pass

# TODO: Test your advanced workflow

<details>
<summary>Click to see solution for Exercise 4</summary>

```python
class RateLimitedProducer(Node):
    def __init__(self, messages_per_second=2, max_messages=20):
        super().__init__()
        self.rate_limit = messages_per_second
        self.max_messages = max_messages
        self.message_count = 0
        self.last_production_time = 0

    async def process(self, context):
        print(f"RateLimitedProducer: Starting at {self.rate_limit} msg/sec")
        
        while not self._stop_flag and self.message_count < self.max_messages:
            # Rate limiting logic
            current_time = time.time()
            time_since_last = current_time - self.last_production_time
            min_interval = 1.0 / self.rate_limit
            
            if time_since_last < min_interval:
                await asyncio.sleep(min_interval - time_since_last)
            
            # Produce message
            self.message_count += 1
            message = {
                'id': self.message_count,
                'content': f'rate_limited_msg_{self.message_count}',
                'timestamp': time.time(),
                'priority': 'high' if self.message_count % 4 == 0 else 'normal'
            }
            
            self.last_production_time = time.time()
            print(f"RateLimitedProducer: {message['id']} (rate: {self.rate_limit}/s)")
            
            yield NodeMessage(content=message)
        
        print(f"RateLimitedProducer: Finished ({self.message_count} messages)")

class BatchProcessor(Node):
    def __init__(self, batch_size=5, timeout=2.0):
        super().__init__()
        self.batch_size = batch_size
        self.timeout = timeout
        self.batch = []
        self.last_flush_time = time.time()
        self.processed_batches = 0

    async def process(self, context):
        message = context.inputs.content
        
        current_time = time.time()
        self.batch.append(message)
        
        print(f"BatchProcessor: Collected {message['id']} (batch: {len(self.batch)}/{self.batch_size})")
        
        # Check if batch should be flushed
        should_flush = (
            len(self.batch) >= self.batch_size or
            (self.batch and current_time - self.last_flush_time >= self.timeout)
        )
        
        if should_flush and self.batch:
            self.processed_batches += 1
            batch_result = {
                'batch_id': self.processed_batches,
                'messages': self.batch.copy(),
                'count': len(self.batch),
                'processed_at': current_time,
                'high_priority_count': sum(1 for msg in self.batch if msg.get('priority') == 'high')
            }
            
            print(f"BatchProcessor: Processing batch {self.processed_batches} ({len(self.batch)} messages)")
            
            # Clear batch and update timestamp
            self.batch.clear()
            self.last_flush_time = current_time
            
            # Update graph state with health info
            await context.graph_state.set('last_batch_time', current_time)
            await context.graph_state.set('total_batches_processed', self.processed_batches)
            
            yield NodeMessage(content=batch_result, metadata={'batch_id': self.processed_batches})

class HealthMonitor(Node):
    def __init__(self, health_check_interval=1.5, alert_threshold=10):
        super().__init__()
        self.check_interval = health_check_interval
        self.alert_threshold = alert_threshold
        self.start_time = time.time()
        self.last_health_time = time.time()
        self.messages_per_window = 0

    async def process(self, context):
        batch_data = context.inputs.content
        current_time = time.time()
        
        self.messages_per_window += batch_data['count']
        time_since_last_check = current_time - self.last_health_time
        
        if time_since_last_check >= self.check_interval:
            # Calculate health metrics
            total_runtime = current_time - self.start_time
            total_batches = await context.graph_state.get('total_batches_processed', 0)
            avg_batch_size = self.messages_per_window / max(1, time_since_last_check)
            
            health_status = {
                'timestamp': current_time,
                'total_runtime': total_runtime,
                'total_batches': total_batches,
                'avg_messages_per_sec': avg_batch_size,
                'current_batch': batch_data['batch_id'],
                'status': 'healthy'
            }
            
            # Check for health issues
            if avg_batch_size < 1.0:  # Low processing rate
                health_status['status'] = 'slow'
                health_status['alert'] = f'Low processing rate: {avg_batch_size:.2f} msg/s'
            elif batch_data['batch_id'] > self.alert_threshold:
                health_status['status'] = 'warning'
                health_status['alert'] = f'High batch number: {batch_data['batch_id']}'
            
            print(f"HealthMonitor: {health_status['status'].upper()} - {health_status}")
            
            if 'alert' in health_status:
                alert_message = {
                    'type': 'health_alert',
                    'alert': health_status['alert'],
                    'timestamp': current_time,
                    'health_data': health_status
                }
                yield NodeMessage(content=alert_message, metadata={'alert': True})
            
            # Reset counters
            self.messages_per_window = 0
            self.last_health_time = current_time
            
            # Update health state
            await context.graph_state.set('last_health_check', current_time)
            await context.graph_state.set('current_health_status', health_status['status'])

async def demo_advanced_workflow():
    print("=== Advanced Long-Running Workflow Demo ===")
    
    # Create nodes
    producer = RateLimitedProducer(messages_per_second=3, max_messages=25)
    batch_processor = BatchProcessor(batch_size=4, timeout=1.5)
    health_monitor = HealthMonitor(health_check_interval=1.0, alert_threshold=15)
    
    # Connect nodes with conditional routing
    producer.goto(batch_processor, condition=lambda outputs: outputs is not None)
    batch_processor.goto(health_monitor, condition=lambda outputs: outputs is not None)
    
    # Create graph
    graph = Graph(
        start=producer,
        initial_state={
            'workflow_name': 'advanced_demo',
            'start_time': time.time(),
            'health_checks_enabled': True
        }
    )
    
    # Create task with signal handling
    task = Task(
        inputs=NodeMessage(content={'start': True}),
        type=TaskType.LONG_RUNNING,
        budget={'max_seconds': 10},
        task_id="advanced_workflow_demo"
    )
    
    # Setup signal handler for graceful shutdown
    shutdown_requested = asyncio.Event()
    
    def signal_handler(signum, frame):
        print(f"\nReceived signal {signum}, initiating graceful shutdown...")
        shutdown_requested.set()
    
    signal.signal(signal.SIGINT, signal_handler)
    signal.signal(signal.SIGTERM, signal_handler)
    
    try:
        # Run with timeout or shutdown signal
        result = await asyncio.wait_for(graph.run(task), timeout=10.0)
        print("Workflow completed normally")
    except asyncio.TimeoutError:
        print("Workflow completed due to timeout")
    except KeyboardInterrupt:
        print("Workflow interrupted by user")
    finally:
        # Stop all nodes
        producer.stop()
        print("All nodes stopped gracefully")
    
    return result

# Run the advanced workflow
# await demo_advanced_workflow()
```

</details>

## Section 5: Practical Examples

### Real-World Use Case: Log Processing Pipeline

A common long-running workflow is log processing:

```python
class LogCollector(Node):
    """Collects logs from multiple sources."""
    async def process(self, context):
        while not self._stop_flag:
            # Collect from files, APIs, etc.
            logs = await collect_logs()
            
            for log_entry in logs:
                yield NodeMessage(content={
                    'timestamp': log_entry.timestamp,
                    'level': log_entry.level,
                    'message': log_entry.message,
                    'source': log_entry.source
                })
            
            await asyncio.sleep(5)  # Check every 5 seconds

class LogParser(Node):
    """Parses and normalizes log entries."""
    async def process(self, context):
        log_entry = context.inputs.content
        
        # Parse and structure the log
        parsed = parse_log(log_entry['message'])
        
        if parsed:
            yield NodeMessage(content={
                'original': log_entry,
                'parsed': parsed,
                'timestamp': log_entry['timestamp']
            })

class LogAnalyzer(Node):
    """Analyzes logs for patterns and anomalies."""
    async def process(self, context):
        parsed_log = context.inputs.content
        
        # Analyze for errors, patterns, etc.
        if is_error(parsed_log['parsed']):
            # Trigger alert
            await self._send_alert(parsed_log)
        
        # Store in time-series database
        await self._store_metrics(parsed_log)

class LogStorer(Node):
    """Stores processed logs efficiently."""
    def __init__(self, batch_size=100):
        super().__init__()
        self.batch = []
        self.batch_size = batch_size

    async def process(self, context):
        log_entry = context.inputs.content
        self.batch.append(log_entry)
        
        if len(self.batch) >= self.batch_size:
            await self._flush_batch()
    
    async def _flush_batch(self):
        # Store batch in database
        await self._database.insert_many(self.batch)
        self.batch.clear()
```

### Real-World Use Case: Message Queue Consumer

```python
class MessageQueueConsumer(Node):
    """Consumes messages from external queue systems."""
    def __init__(self, queue_url, max_workers=5):
        super().__init__()
        self.queue_url = queue_url
        self.max_workers = max_workers
        self.workers = []

    async def process(self, context):
        # Start multiple workers
        for i in range(self.max_workers):
            worker = asyncio.create_task(self._worker_loop(f"worker_{i}"))
            self.workers.append(worker)
        
        # Wait for stop signal or workers to finish
        while not self._stop_flag:
            await asyncio.sleep(0.1)
        
        # Wait for all workers to finish current messages
        await asyncio.gather(*self.workers, return_exceptions=True)
    
    async def _worker_loop(self, worker_id):
        while not self._stop_flag:
            try:
                # Get message from queue (with timeout)
                message = await self._queue.receive(timeout=1.0)
                
                # Process message and yield result
                processed = await self._process_message(message)
                yield NodeMessage(content=processed, metadata={'worker': worker_id})
                
            except asyncio.TimeoutError:
                # No message available, continue
                continue
            except Exception as e:
                print(f"Worker {worker_id} error: {e}")
                await asyncio.sleep(1.0)  # Back off on error
```

### Monitoring and Observability

```python
class MetricsCollector(Node):
    """Collects and reports workflow metrics."""
    def __init__(self, reporting_interval=10.0):
        super().__init__()
        self.reporting_interval = reporting_interval
        self.start_time = time.time()
        self.message_count = 0
        self.error_count = 0

    async def process(self, context):
        # Update counters
        self.message_count += 1
        
        if context.inputs.metadata.get('error'):
            self.error_count += 1
        
        # Report metrics periodically
        if time.time() - self.start_time >= self.reporting_interval:
            await self._report_metrics()
            self.start_time = time.time()

    async def _report_metrics(self):
        metrics = {
            'messages_processed': self.message_count,
            'errors': self.error_count,
            'error_rate': self.error_count / max(1, self.message_count),
            'uptime': time.time() - self.start_time
        }
        
        # Send to monitoring system
        await self._monitoring_system.report(metrics)
```

### Exercise 5: Build a Complete Long-Running Application

Create a complete long-running application with:

1. **Twitter-style Feed Processor**: Processes incoming social media posts
2. **Content Analyzer**: Analyzes sentiment and extracts keywords
3. **Trend Detector**: Identifies trending topics
4. **Alert Generator**: Sends alerts for important content
5. **Metrics Dashboard**: Tracks processing statistics

Use all patterns covered: persistence, batching, rate limiting, monitoring, and graceful shutdown.

In [ ]:
# Your solution for Exercise 5 - Complete Long-Running Application
# Build a Twitter-style feed processing pipeline

from spark.graphs import Graph, Task, TaskType, SQLiteStateBackend
from spark.nodes import Node
from spark.nodes.types import NodeMessage
from spark.nodes.channels import ChannelMessage
import asyncio
import time
import random
from pathlib import Path

# TODO: Implement all components of the social media processing pipeline

class FeedProducer(Node):
    """Simulates incoming social media posts"""
    # TODO: Implement feed production
    pass

class ContentAnalyzer(Node):
    """Analyzes sentiment and extracts keywords"""
    # TODO: Implement content analysis
    pass

class TrendDetector(Node):
    """Identifies trending topics"""
    # TODO: Implement trend detection
    pass

class AlertGenerator(Node):
    """Sends alerts for important content"""
    # TODO: Implement alert generation
    pass

class MetricsDashboard(Node):
    """Tracks processing statistics"""
    # TODO: Implement metrics collection
    pass

async def build_social_media_pipeline():
    """Build and run the complete social media processing pipeline"""
    # TODO: Create complete pipeline with all components
    pass

# TODO: Test your complete application

## Summary

### Key Concepts Covered

1. **Long-Running Execution**: `TaskType.LONG_RUNNING` enables concurrent node execution
2. **Channel Communication**: Nodes communicate via persistent mailboxes instead of direct calls
3. **Message Patterns**: Support for pub/sub, fan-out, fan-in, and priority routing
4. **Persistence**: Mailboxes and GraphState survive process restarts
5. **Streaming**: Continuous data processing with `yield` patterns
6. **Backpressure**: Manage flow between fast producers and slow consumers
7. **Graceful Shutdown**: Clean resource cleanup and signal handling

### When to Use Long-Running Workflows

✅ **Good for**: Real-time data processing, event-driven systems, continuous monitoring, message queue consumers

❌ **Not for**: One-time data transformations, batch ETL jobs, simple request/response patterns

### Best Practices

1. **Use Persistence**: Always use `SQLiteStateBackend` or similar for production workflows
2. **Handle Backpressure**: Implement rate limiting and queue size monitoring
3. **Graceful Shutdown**: Always handle `SIGINT`/`SIGTERM` signals
4. **Monitor Health**: Track message rates, error rates, and system health
5. **Batch Processing**: Group messages for efficiency when appropriate
6. **Resource Management**: Close connections and cleanup resources in `finally` blocks
7. **Observability**: Add logging and metrics for debugging and monitoring

### Performance Considerations

- **Memory Usage**: Large mailboxes consume memory; consider size limits
- **Database Load**: Persistent mailboxes create database I/O; monitor performance
- **Concurrency**: Use appropriate number of workers for your hardware
- **Network Latency**: Account for external service delays in your designs

## Next Steps

After completing this tutorial, you're ready for:

- **Tutorial 11**: Event-Driven Architecture - Advanced event patterns and reactive systems
- **Tutorial 12**: Observability with Telemetry - Comprehensive monitoring and metrics
- **Tutorial 13**: Distributed Workflows with RPC - Building distributed systems

## Additional Resources

- **Example Code**: `examples/e010_long_running_workflows.py` - Complete working examples
- **Channel Documentation**: `spark/nodes/channels.py` - Full channel API reference
- **Task Types**: `spark/graphs/tasks.py` - All available task types and configurations
- **State Backend**: `spark/graphs/graph_state.py` - Persistent state management

Congratulations! You've completed Tutorial 10: Long-Running Workflows & Channels. You now have the skills to build sophisticated real-time processing systems with Spark!